# 1) Bed Availability and Occupancy (Overnight) 

## File Used
**Beds-Open-Overnight-Web_File-Q3-2025-26_duplicate.xlsx**

---

## What This File Contains

This dataset provides **average daily overnight bed statistics**, including:

- Total beds available  
- Total beds occupied  
- Percentage of beds occupied  

The data is broken down by:
- **Bed sector** (e.g., General & Acute, Mental Health, etc.)
- **Region**
- An overall **England total**

This dataset is commonly used to assess **hospital capacity pressure**.

---

## What "Acute" Means

In this context, **Acute** refers to patients who require:

- Short-term, active medical treatment  
- Urgent or emergency care  
- Surgery or intensive monitoring  
- Treatment for serious illness or injury  

### General & Acute Beds

**General & Acute beds** typically include beds used for:

- Emergency admissions (A&E admissions)
- Non-elective urgent cases
- Elective (planned) surgery recovery
- General medical wards
- Surgical wards

They do **not** usually include:
- Long-term care beds
- Mental health beds
- Maternity-only beds
- Community hospital beds

Because General & Acute beds handle the majority of urgent hospital activity, their **% occupancy** is widely used as a system pressure indicator.

---

## Sheet Used
**Region by Sector**  
(This is the only sheet in the file.)

---

## Key Fields Used

From the sheet:

- **Year** — Reporting year  
- **Period End** — End date of the reporting period  
- **AT Name** — Region name (e.g., England, London, etc.)

---

## Bed Metrics and Column Naming

For each bed type (e.g., General & Acute), the dataset follows this structure:

- **Available beds** → `General & Acute`
- **Occupied beds** → `General & Acute.1`
- **% Occupied** → `General & Acute.2`

The same naming pattern applies to other bed sectors.

---

## Fields Primarily Used for the Project

The analysis primarily focuses on:

### 1. England Total
Filtered using:
```
AT Name == "England"
```

### 2. General & Acute % Occupancy
This metric is used as the **primary pressure signal** because:

- High occupancy (>85%) indicates system strain  
- Sustained occupancy above safe thresholds reduces surge capacity  
- It directly reflects real-time hospital operational pressure  

---

## Why This Dataset Matters

Monitoring General & Acute overnight occupancy allows:

- Tracking system stress over time  
- Comparing regional performance  
- Identifying winter pressure periods  
- Supporting capacity planning and policy analysis  

# 1.0 Data Cleaning (Beds – Overnight Occupancy)

## Objective
Prepare the NHS England **Bed Availability and Occupancy (Overnight)** dataset for analysis by:
- removing metadata/empty rows and irrelevant columns,
- selecting the **national-level (England)** record,
- standardising column names,
- creating a clean **datetime** field for merging with A&E monthly data.

## Why this matters
The raw Excel sheet contains:
- multiple regions (England + regions),
- multiple bed categories (General & Acute, Maternity, etc.),
- repeated column names that pandas labels as `.1` and `.2`.

For this project, we focus on **General & Acute beds**, because they are the primary capacity constraint influencing emergency flow.

## Output
A clean dataframe called `beds_clean` with:
- `period_date` (datetime)
- `ga_available`
- `ga_occupied`
- `ga_occupancy_rate`
Ready for merging with the A&E dataset.

In [1]:
#Importing operating system helper
import os


In [2]:
#Finding my notebook directory
os.getcwd()


'C:\\Users\\folah\\my_python_projects'

In [3]:

os.listdir()


['.ipynb_checkpoints',
 'Beds-Open-Overnight-Web_File-Q3-2025-26_duplicate.xlsx',
 'Monthly-AE-Time-Series-January-2026-C86cfU_duplicate.xls',
 'NHSDATA.ipynb',
 'venv']

In [4]:
import pandas as pd



In [5]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:


beds = pd.read_excel(
    "Beds-Open-Overnight-Web_File-Q3-2025-26_duplicate.xlsx",
    sheet_name="Region by Sector",
    header=14
)

beds.head()

,Unnamed: 0,Year,Period End,Region Code,AT Name,Total,General & Acute,Learning Disability,Maternity,Mental Illness,...,General & Acute.1,Learning Disability.1,Maternity.1,Mental Illness.1,Unnamed: 16,Total .2,General & Acute.2,Learning Disability.2,Maternity.2,Mental Illness.2
0,NaN,2025-26,December,NaN,England,129905.358696,103824.239130,735.771739,7291.250000,18054.097826,...,94968.956522,452.358696,4466.163043,16095.043478,NaN,0.892823,0.914709,0.614808,0.612537,0.891490
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,2025-26,December,Y56,LONDON,20694.597826,15654.271739,44.293478,1276.163043,3719.869565,...,14254.282609,31.630435,838.750000,3394.630435,NaN,0.894885,0.910568,0.714110,0.657244,0.912567
3,NaN,2025-26,December,Y58,SOUTH WEST,12072.043478,10089.641304,42.695652,608.543478,1331.163043,...,9474.032609,17.586957,360.271739,1203.478261,NaN,0.915783,0.938986,0.411914,0.592023,0.904080
4,NaN,2025-26,December,Y59,SOUTH EAST,17585.663043,14290.304348,47.717391,1037.521739,2210.119565,...,13068.369565,24.228261,592.597826,1988.739130,NaN,0.891291,0.914492,0.507745,0.571167,0.899833


In [7]:
# Inspect the structure (rows and column of the dataset) and  columns
print('Intial Structure', beds.shape)
print("Columns:", beds.columns.tolist())


Intial Structure (9, 22)
Columns: ['Unnamed: 0', 'Year', 'Period End', 'Region Code', 'AT Name', 'Total ', 'General & Acute', 'Learning Disability', 'Maternity', 'Mental Illness', 'Unnamed: 10', 'Total .1', 'General & Acute.1', 'Learning Disability.1', 'Maternity.1', 'Mental Illness.1', 'Unnamed: 16', 'Total .2', 'General & Acute.2', 'Learning Disability.2', 'Maternity.2', 'Mental Illness.2']


In [8]:
beds= beds.dropna(how='all').copy()
print('After dropping empty rows:', beds.shape)
beds.head(3)

After dropping empty rows: (8, 22)


,Unnamed: 0,Year,Period End,Region Code,AT Name,Total,General & Acute,Learning Disability,Maternity,Mental Illness,...,General & Acute.1,Learning Disability.1,Maternity.1,Mental Illness.1,Unnamed: 16,Total .2,General & Acute.2,Learning Disability.2,Maternity.2,Mental Illness.2
0,NaN,2025-26,December,NaN,England,129905.358696,103824.239130,735.771739,7291.250000,18054.097826,...,94968.956522,452.358696,4466.163043,16095.043478,NaN,0.892823,0.914709,0.614808,0.612537,0.891490
2,NaN,2025-26,December,Y56,LONDON,20694.597826,15654.271739,44.293478,1276.163043,3719.869565,...,14254.282609,31.630435,838.750000,3394.630435,NaN,0.894885,0.910568,0.714110,0.657244,0.912567
3,NaN,2025-26,December,Y58,SOUTH WEST,12072.043478,10089.641304,42.695652,608.543478,1331.163043,...,9474.032609,17.586957,360.271739,1203.478261,NaN,0.915783,0.938986,0.411914,0.592023,0.904080


In [9]:
#look for columns that starts with unnamed

Unnamed_cols = [c for c in beds.columns if str(c).startswith('Unnamed')]
#drop columns if they exist
if Unnamed_cols:
    beds = beds.drop(columns=Unnamed_cols)

print('Dropped Unnamed columns:', Unnamed_cols)

Dropped Unnamed columns: ['Unnamed: 0', 'Unnamed: 10', 'Unnamed: 16']


In [10]:
beds.columns.tolist()
beds.head()

,Year,Period End,Region Code,AT Name,Total,General & Acute,Learning Disability,Maternity,Mental Illness,Total .1,General & Acute.1,Learning Disability.1,Maternity.1,Mental Illness.1,Total .2,General & Acute.2,Learning Disability.2,Maternity.2,Mental Illness.2
0,2025-26,December,NaN,England,129905.358696,103824.239130,735.771739,7291.250000,18054.097826,115982.521739,94968.956522,452.358696,4466.163043,16095.043478,0.892823,0.914709,0.614808,0.612537,0.891490
2,2025-26,December,Y56,LONDON,20694.597826,15654.271739,44.293478,1276.163043,3719.869565,18519.293478,14254.282609,31.630435,838.750000,3394.630435,0.894885,0.910568,0.714110,0.657244,0.912567
3,2025-26,December,Y58,SOUTH WEST,12072.043478,10089.641304,42.695652,608.543478,1331.163043,11055.369565,9474.032609,17.586957,360.271739,1203.478261,0.915783,0.938986,0.411914,0.592023,0.904080
4,2025-26,December,Y59,SOUTH EAST,17585.663043,14290.304348,47.717391,1037.521739,2210.119565,15673.934783,13068.369565,24.228261,592.597826,1988.739130,0.891291,0.914492,0.507745,0.571167,0.899833
5,2025-26,December,Y60,MIDLANDS,23709.663043,18986.978261,188.869565,1311.869565,3221.945652,21227.706522,17420.010870,124.847826,875.586957,2807.260870,0.895319,0.917471,0.661027,0.667434,0.871294


In [11]:
# Note:
# NaN in 'Region Code' does NOT mean missing data.
# England represents the national aggregate and does not belong to any region.
#
# The following code (NOT executed) would replace NaN values
# with an empty string if required:
beds.loc[beds['Region Code'].isnull(), 'Region Code'] = 'Eng'
beds['Region Code']

0    Eng
2    Y56
3    Y58
4    Y59
5    Y60
6    Y61
7    Y62
8    Y63
Name: Region Code, dtype: str

In [12]:
#Standardisation

beds['AT Name'] = beds['AT Name'].str.upper()
beds['AT Name']

0                     ENGLAND
2                      LONDON
3                  SOUTH WEST
4                  SOUTH EAST
5                    MIDLANDS
6             EAST OF ENGLAND
7                  NORTH WEST
8    NORTH EAST AND YORKSHIRE
Name: AT Name, dtype: str

In [13]:
#Each key is the original column name (must match exactly, including spaces).
#Each value is the new cleaner, more descriptive column name.

beds = beds.rename(columns={
    "Total ": "beds_available_total",          # total beds available
    "General & Acute": "ga_available",         # general & acute beds available

    "Total .1": "beds_occupied_total",         # total beds occupied
    "General & Acute.1": "ga_occupied",        # general & acute beds occupied

    "Total .2": "beds_occupancy_rate",         # overall occupancy rate
    "General & Acute.2": "ga_occupancy_rate"   # general & acute occupancy rate
})
beds.head()

,Year,Period End,Region Code,AT Name,beds_available_total,ga_available,Learning Disability,Maternity,Mental Illness,beds_occupied_total,ga_occupied,Learning Disability.1,Maternity.1,Mental Illness.1,beds_occupancy_rate,ga_occupancy_rate,Learning Disability.2,Maternity.2,Mental Illness.2
0,2025-26,December,Eng,ENGLAND,129905.358696,103824.239130,735.771739,7291.250000,18054.097826,115982.521739,94968.956522,452.358696,4466.163043,16095.043478,0.892823,0.914709,0.614808,0.612537,0.891490
2,2025-26,December,Y56,LONDON,20694.597826,15654.271739,44.293478,1276.163043,3719.869565,18519.293478,14254.282609,31.630435,838.750000,3394.630435,0.894885,0.910568,0.714110,0.657244,0.912567
3,2025-26,December,Y58,SOUTH WEST,12072.043478,10089.641304,42.695652,608.543478,1331.163043,11055.369565,9474.032609,17.586957,360.271739,1203.478261,0.915783,0.938986,0.411914,0.592023,0.904080
4,2025-26,December,Y59,SOUTH EAST,17585.663043,14290.304348,47.717391,1037.521739,2210.119565,15673.934783,13068.369565,24.228261,592.597826,1988.739130,0.891291,0.914492,0.507745,0.571167,0.899833
5,2025-26,December,Y60,MIDLANDS,23709.663043,18986.978261,188.869565,1311.869565,3221.945652,21227.706522,17420.010870,124.847826,875.586957,2807.260870,0.895319,0.917471,0.661027,0.667434,0.871294
